In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3
import datetime as dt

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2024-03-25 09:06:32.538799


### Functions

In [3]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [4]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
# sub task
str_subtask = os.getcwd().split('\\')[6]
print(f'Subtask: {str_subtask}')
# output
str_dirname_output = './output'

Project: 20231010-gen-xii
Task: 12_dark_scoring
Subtask: 01_pull_scores_from_db


### Output directory

In [5]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

### Read query

In [6]:
str_filepath = './sql/query.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('with tbl1 as\n'
 '(\n'
 'select\n'
 'tblAccount.bigAccountId,\n'
 'tblAccount.dtmStampCreation, \n'
 'tblaccount.dtmApproved, \n'
 'tblAccount.dtmFunded,\n'
 'strScoreCardVersion, \n'
 'fltDebtorScore,\n'
 'fltDebtor_Score_lgd,\n'
 'fltDebtor_Score_pd,\n'
 'fltDebtor_Score_ad,\n'
 'Row_number() OVER(partition BY intAccountKey, strScoreCardVersion ORDER BY '
 'DimScoreCard.dtmStampCreation desc) RowNum\n'
 'from tblAccount left outer join edw.pfsedw.dbo.DimScoreCard\n'
 'on tblAccount.bigaccountid = DimScoreCard.intAccountKey\n'
 "--where dtmStampCreation >= '2024-01-01'\n"
 ')\n'
 'select*\n'
 'from tbl1\n'
 'where RowNum = 1\n'
 "and strScoreCardVersion like 'genxii_v2'\n"
 "and tbl1.dtmStampCreation >= '2024-03-14'")


### Write into df

In [7]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()

# show
df

Wall time: 3 s


,bigAccountId,dtmStampCreation,dtmApproved,dtmFunded,strScoreCardVersion,fltDebtorScore,fltDebtor_Score_lgd,fltDebtor_Score_pd,fltDebtor_Score_ad,RowNum
0,7657064,2024-03-14 05:58:34.890,2024-03-14 05:58:54.893,NaT,genxii_v2,0.143229,0.366327,0.390988,0.054941,1
1,7657065,2024-03-14 06:19:01.423,NaT,NaT,genxii_v2,0.008224,0.300804,0.027340,0.085813,1
2,7657070,2024-03-14 06:52:05.273,2024-03-14 06:52:19.663,NaT,genxii_v2,0.130835,0.389506,0.335900,0.334481,1
3,7657071,2024-03-14 06:56:01.100,NaT,NaT,genxii_v2,0.138227,0.418457,0.330326,0.445710,1
4,7657072,2024-03-14 06:57:17.540,NaT,NaT,genxii_v2,0.242750,0.584907,0.535794,0.407639,1
...,...,...,...,...,...,...,...,...,...,...
17714,7704331,2024-03-24 20:13:01.587,NaT,NaT,genxii_v2,0.147814,0.485824,0.304254,0.215153,1
17715,7704333,2024-03-24 20:47:55.387,2024-03-24 20:48:17.837,NaT,genxii_v2,0.066877,0.448009,0.149277,0.261241,1
17716,7704334,2024-03-24 20:51:22.957,NaT,NaT,genxii_v2,0.182442,0.450335,0.405126,0.383911,1
17717,7704336,2024-03-24 21:06:41.647,2024-03-24 21:07:09.940,NaT,genxii_v2,0.043904,0.392720,0.111795,0.196025,1


### Save as parquet

In [8]:
%%time

# save
str_filename = 'df_scores.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_parquet(str_local_path, compression='gzip')

Wall time: 235 ms


### Upload to s3

In [9]:
%%time

# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'{str_task}/{str_subtask}/{str_filename}', 
    str_bucket_name=str_project,
)

Wall time: 592 ms


### Clean-up

In [10]:
os.remove(str_local_path)